# Database Persistence
* Currently, every time you run the script, it reads the CSV files from scratch, does the math, prints the result, and forgets everything the second the script finishes.
* If this ran in a real FinTech company every morning, it would re-reconcile the exact same transactions from three years ago, every single day.
* To fix this, we need State Management. We are going to replace our static Internal_Ledger.csv with a real Relational Database.

In [1]:
pip install SQLAlchemy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 8.1 MB/s  0:00:00 eta 0:00:01
Note: you may need to restart the kernel to use updated packages.


In our CSV version, data was just a flat text file. If we wanted to change a transaction's status from "Pending" to "Reconciled," we had to rewrite the entire file. That is incredibly dangerous and slow. By moving to a Relational Database (like SQLite, PostgreSQL, or MySQL), we get State Management. <br>

Things to tackle: 
1. The Engine: This is the literal connection bridge between your Python script and the database file.
2. The Table (__tablename__): A database is made of tables (like tabs in Excel). We are going to build a table called internal_ledger.
3. The Schema (Columns & Data Types): Unlike a CSV where anything goes, a database is strictly typed. If we define the amount column as a Float, the database will aggressively reject any text strings. This protects our data integrity.
4. The Primary Key (primary_key=True): Every row in a database MUST have a unique ID number. It is the absolute, unchanging address of that row.
5. The State (status): This is the most important addition. We are adding a column that defaults to "Pending". Later, when our Python engine matches a transaction, it will log into the database and permanently change this column to "Reconciled".



In [ ]:
from sqlalchemy import create_engine, Column, Integer, String, Float, DateTime
from sqlalchemy.orm import declarative_base
import os

# 1. THE ENGINE: The bridge to our SQLite database file
# 'sqlite:///../data/reconciliation.db' tells it to create a file in our data folder
engine = create_engine('sqlite:///../data/reconciliation.db', echo=True)

# 2. THE BASE: The SQLAlchemy template that all our tables will inherit from, it is something you always need to create when using SQLAlchemy
# Python and SQL do not natively understand each other.
    # Python thinks in Objects (Classes, attributes, methods).
    # Databases think in Relations (Tables, columns, rows).
    # If you want to create a database table using Python, you need a translator.
# Think of Base as an empty clipboard. It is a registry that exists to keep track of every single table you want to build in your database.
Base = declarative_base()
# If we didn't use declarative_base(), and you wanted to add 5 new tables to your FinTech app (like Customers, BankAccounts, AuditLogs), 
# you would have to write hundreds of lines of raw SQL code by hand and manage them yourself.
    # With declarative_base(), you just write 5 standard Python classes that inherit from Base, run Base.metadata.create_all(), 
    # and SQLAlchemy builds the entire complex database automatically!

# 3. THE SCHEMA: Defining our table as a Python class
class InternalTransaction(Base):
    # this registers the table in our Base registry so SQLAlchemy knows to create it when we call Base.metadata.create_all(engine) later
    __tablename__ = 'internal_ledger' # The actual name of the table in the database

    # The Primary Key: The database's internal, sequential ID (1, 2, 3...)
    id = Column(Integer, primary_key=True)
    
    # Our App's ID: We enforce unique=True so we can never accidentally insert the same transaction twice!
    transaction_id = Column(String, unique=True, nullable=False)
    
    # The Business Data
    date = Column(DateTime, nullable=False)
    customer_name = Column(String)
    amount = Column(Float, nullable=False)
    gateway = Column(String)
    
    # THE STATE: This is how we prevent the engine from re-processing old data
    status = Column(String, default="Pending")

# 4. THE BUILDER: Tell the engine to physically create the tables in the file
def init_db():
    print("Connecting to database and building schema...")
    # This translates our Python class into a SQL "CREATE TABLE" command
    Base.metadata.create_all(engine)
    print("Success! Database schema created.")

# if __name__ == "__main__":
#     init_db()

In the code for class declaration you're essentially writing:

CREATE TABLE internal_ledger (
    id INTEGER PRIMARY KEY,
    transaction_id VARCHAR NOT NULL UNIQUE,
    date DATETIME NOT NULL,
    customer_name VARCHAR,
    amount FLOAT NOT NULL,
    gateway VARCHAR,
    status VARCHAR DEFAULT 'Pending'
);

# Seeding the database
* "Seeding" is the industry term for populating an empty database with its initial batch of data. We are going to take the thousands of rows from our Batched_Internal_Ledger.csv and securely inject them into our new SQLite tables.

Things to do:

You have to open a Session and use Transactions.
1. The Session: This is your temporary workspace. Think of it like a waiting room. When we read our CSV, we will create Python objects for each row and put them in this waiting room.
2. The Commit: Once all the records are in the waiting room, we call session.commit(). This translates to a massive SQL INSERT command. The database takes everything in the waiting room and permanently writes it to the hard drive in one fell swoop.
3. The Rollback (ACID Compliance): What if row #4,999 has a duplicate Transaction_ID? Because we set unique=True in our schema, the database will throw an error. Instead of leaving 4,998 rows in the database and failing on the rest, a transaction guarantees an "all-or-nothing" approach. We call session.rollback(), and the database automatically undoes the entire operation, keeping your data perfectly clean.

REFERENCE [seed_db.py]() for comments and explanation

# Reconciling at the DB level
We want to only try and reconcile those that have status == "Pending" (the default)

Steps:
1. READ only the rows where status == "Pending".
2. MATCH them against the bank statement using our pandas logic.
3. UPDATE the exact rows in the database to status = "Reconciled".

REFERENCE [reconcile_db.py]() for comments and explanations